<a href="https://colab.research.google.com/github/maria-gigi/Diabetes-IA/blob/main/Diabetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler as ss

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
data = pd.read_csv("diabetes.csv")
print(data)

     Number of times pregnant  Plasma glucose concentration  \
0                           6                           148   
1                           1                            85   
2                           8                           183   
3                           1                            89   
4                           0                           137   
..                        ...                           ...   
763                        10                           101   
764                         2                           122   
765                         5                           121   
766                         1                           126   
767                         1                            93   

     Diastolic blood pressure  Triceps skin fold thickness  \
0                          72                           35   
1                          66                           29   
2                          64                            

In [ ]:
#x: necessário transformar os valores da tabela em arrays para aplicar na rede neural
#y: coluna que distingue quem é diabético e quem não é
x = data.iloc[:,0:-1].values #iloc remove a última coluna da tabela
y_string = list(data.iloc[:, -1]) #pega apenas a última coluna

In [ ]:
#converter string para int, pois a rede entende apenas números
y_int = []
for string in y_string:
  if string == "positive":
    y_int.append(1)
  else:
    y_int.append(0)
#convertendo y para array:
y = np.array(y_int, dtype = "float64")

In [ ]:
#Necessário normalizar os dados pois iremos utilizar o método de SVM, que utiliza a distância Euclidiana
sc = ss()
x = sc.fit_transform(x)

In [ ]:
x = torch.tensor(x)
y = torch.tensor(y).unsqueeze(1)

In [ ]:
class Dataset(Dataset):
  def __init__(self, x, y):
    self.x = x
    self.y = y
  def __getitem__(self, index):
    return self.x[index], self.y[index]
  def __len__(self):
    return len(self.x)

In [ ]:
dataset = Dataset(x, y)

In [ ]:
train_loader = DataLoader(dataset = dataset,
                          batch_size = 32,
                          shuffle = True)

In [ ]:
print("There is {} batches in dataset".format(len(train_loader)))
for (x, y) in train_loader:
  print("For one interation (batch) there is:")
  print("Data:    {}".format(x.shape))
  print("Labels:  {}".format(y.shape))
  break

There is 24 batches in dataset
For one interation (batch) there is:
Data:    torch.Size([32, 7])
Labels:  torch.Size([32, 1])


In [ ]:
class Model(nn.Module):
  def __init__(self, input_features, output_features):
    super(Model, self).__init__()
    self.fc1 = nn.Linear(input_features, 5)
    self.fc2 = nn.Linear(5, 4)
    self.fc3 = nn.Linear(4, 3)
    self.fc4 = nn.Linear(3, output_features)
    self.sigmoid = nn.Sigmoid()
    self.tanh = nn.Tanh()

  def forward(self, x):
    out = self.fc1(x)
    out = self.tanh(out)
    out = self.fc2(out)
    out = self.tanh(out)
    out = self.fc3(out)
    out = self.tanh(out)
    out = self.fc4(out)
    out = self.sigmoid(out)
    return out

In [ ]:
net = Model(7, 1)

criterion = torch.nn.BNELoss(size_average = True)
optimizer = torch.optim.SGD(net.parameters(), lr = 0.1, momentum = 0.9)

In [ ]:
epochs = 200
for epoch in range(200):
  for inputs, labels in train_loader:
    inputs = inputs.float()
    labels = labels.float()
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step() #atualizar pesos

  #Cálculo de acurácia
  output = (output > 0.5).float()
  accuracy = (output == labels).mean()
  print("Epoch {}/{}, Loss: {:, 3f}, Accuracy: {:, 3f}".format(epoch + 1, epochs, loss, accuracy))
